<a href="https://colab.research.google.com/github/AksharaBasu/GenAI/blob/main/6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata

LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")

print("LANGSMITH_API_KEY : FOUND")

LANGSMITH_API_KEY : FOUND


In [ ]:
# ============================================================
# INSTALL
# ============================================================
!pip install -q -U langgraph langchain-openrouter langchain-groq langsmith gradio


# ============================================================
# IMPORTS
# ============================================================
import os
import gradio as gr
from typing import TypedDict
from google.colab import userdata

from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END


# ============================================================
# GET API KEYS
# ============================================================
OPENROUTER_API_KEY = userdata.get("openrouter")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
AKSHARA_API_KEY = userdata.get("akshara")
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")


# ============================================================
# API KEY STATUS
# ============================================================
print("=" * 36)
print("API KEY STATUS")
print("=" * 36)

print(f"openrouter        : {'FOUND' if OPENROUTER_API_KEY else 'MISSING'}")
print(f"GROQ_API_KEY      : {'FOUND' if GROQ_API_KEY else 'MISSING'}")
print(f"akshara           : {'FOUND' if AKSHARA_API_KEY else 'MISSING'}")
print(f"LANGSMITH_API_KEY : {'FOUND' if LANGSMITH_API_KEY else 'MISSING'}")


# ============================================================
# LANGSMITH
# ============================================================
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "3-LLM-LangGraph-Generate-Check-Backup"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

print()
print("=" * 36)
print("LANGSMITH")
print("=" * 36)
print("Tracing: ENABLED")
print("Project: 3-LLM-LangGraph-Generate-Check-Backup")


# ============================================================
# CREATE 3 LLMs
# ============================================================

# LLM 1 — GENERATE
llm1 = ChatOpenRouter(
    model="openai/gpt-oss-20b",
    api_key=OPENROUTER_API_KEY,
    temperature=0
)

# LLM 2 — CHECK / CORRECT
llm2 = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

# LLM 3 — BACKUP
llm3 = ChatOpenRouter(
    model="qwen/qwen3-32b",
    api_key=AKSHARA_API_KEY,
    temperature=0
)

print()
print("LLMs")
print("LLM 1: READY")
print("LLM 2: READY")
print("LLM 3: READY")


# ============================================================
# LANGGRAPH STATE
# ============================================================
class State(TypedDict, total=False):

    question: str

    llm1_answer: str
    llm2_answer: str
    llm3_answer: str

    check_result: str

    final_answer: str

    selected_llm: str

    error_llm1: str
    error_llm2: str
    error_llm3: str


# ============================================================
# LLM 1 NODE
# ============================================================
def llm1_node(state: State):

    question = state["question"]

    try:

        # Special test:
        # forces LLM 1 to fail
        if "FORCE_LLM1_LLM2_FAILURE_TEST" in question:

            return {
                "llm1_answer": "",
                "error_llm1": "Forced test failure"
            }

        prompt = f"""
Answer the following question accurately and clearly.

Question:
{question}
"""

        response = llm1.invoke(prompt)

        answer = response.content

        if not answer:
            raise Exception("LLM 1 returned empty answer")

        return {
            "llm1_answer": answer,
            "selected_llm": "LLM 1"
        }

    except Exception as e:

        return {
            "llm1_answer": "",
            "error_llm1": str(e)
        }


# ============================================================
# ROUTE AFTER LLM 1
# ============================================================
def route_after_llm1(state: State):

    return "llm2"


# ============================================================
# LLM 2 NODE
# ============================================================
def llm2_node(state: State):

    question = state["question"]
    llm1_answer = state.get("llm1_answer", "")

    try:

        # Special test:
        # forces LLM 2 to fail
        if "FORCE_LLM1_LLM2_FAILURE_TEST" in question:

            return {
                "llm2_answer": "",
                "check_result": "FAILED",
                "error_llm2": "Forced test failure"
            }

        # If LLM 1 failed normally
        if not llm1_answer:

            prompt = f"""
LLM 1 failed to answer.

Answer this question yourself accurately.

Question:
{question}
"""

            response = llm2.invoke(prompt)

            answer = response.content

            if not answer:
                raise Exception("LLM 2 returned empty answer")

            return {
                "llm2_answer": answer,
                "check_result": "CORRECT",
                "selected_llm": "LLM 2"
            }

        # Normal checking
        prompt = f"""
You are a strict answer checker.

Original Question:
{question}

LLM 1 Answer:
{llm1_answer}

Check the answer.

If the answer is correct, start your response with:

CORRECT

If the answer is incorrect, start your response with:

INCORRECT

If incorrect, provide the corrected answer.
"""

        response = llm2.invoke(prompt)

        result = response.content

        if not result:
            raise Exception("LLM 2 returned empty response")

        if result.upper().startswith("CORRECT"):

            return {
                "llm2_answer": result,
                "check_result": "CORRECT"
            }

        else:

            return {
                "llm2_answer": result,
                "check_result": "INCORRECT"
            }

    except Exception as e:

        return {
            "llm2_answer": "",
            "check_result": "FAILED",
            "error_llm2": str(e)
        }


# ============================================================
# ROUTE AFTER LLM 2
# ============================================================
def route_after_llm2(state: State):

    if state.get("check_result") == "FAILED":
        return "llm3"

    return "end"


# ============================================================
# LLM 3 BACKUP NODE
# ============================================================
def llm3_node(state: State):

    question = state["question"]

    try:

        prompt = f"""
You are the emergency backup AI.

LLM 1 and LLM 2 failed.

Answer the following question accurately and clearly.

Question:
{question}
"""

        response = llm3.invoke(prompt)

        answer = response.content

        if not answer:
            raise Exception("LLM 3 returned empty answer")

        return {
            "llm3_answer": answer,
            "final_answer": answer,
            "selected_llm": "LLM 3"
        }

    except Exception as e:

        return {
            "final_answer": "All LLMs failed.",
            "error_llm3": str(e),
            "selected_llm": "NONE"
        }


# ============================================================
# BUILD LANGGRAPH
# ============================================================
print()
print("=" * 36)
print("BUILDING LANGGRAPH")
print("=" * 36)


graph = StateGraph(State)

graph.add_node("llm1", llm1_node)
graph.add_node("llm2", llm2_node)
graph.add_node("llm3", llm3_node)

graph.add_edge(START, "llm1")

graph.add_conditional_edges(
    "llm1",
    route_after_llm1,
    {
        "llm2": "llm2"
    }
)

graph.add_conditional_edges(
    "llm2",
    route_after_llm2,
    {
        "end": END,
        "llm3": "llm3"
    }
)

graph.add_edge("llm3", END)


# ============================================================
# COMPILE
# ============================================================
app = graph.compile()

print("✅ LANGGRAPH COMPILED SUCCESSFULLY")


# ============================================================
# ASK AI
# ============================================================
def ask_ai(question):

    if not question.strip():
        return "Please enter a question."

    try:

        result = app.invoke(
            {
                "question": question
            },
            config={
                "run_name": "3-LLM-LangGraph-Generate-Check-Backup"
            }
        )

        # ----------------------------------------------------
        # LLM 3 BACKUP
        # ----------------------------------------------------
        if result.get("selected_llm") == "LLM 3":

            return (
                "SELECTED LLM: LLM 3 - EMERGENCY BACKUP\n\n"
                "FINAL ANSWER:\n\n"
                + result.get("final_answer", "")
            )

        # ----------------------------------------------------
        # LLM 2 CORRECTED ANSWER
        # ----------------------------------------------------
        if result.get("check_result") == "INCORRECT":

            return (
                "SELECTED LLM: LLM 2\n\n"
                "FINAL ANSWER:\n\n"
                + result.get("llm2_answer", "")
            )

        # ----------------------------------------------------
        # LLM 1 ANSWER
        # ----------------------------------------------------
        if result.get("check_result") == "CORRECT":

            return (
                "SELECTED LLM: LLM 1\n\n"
                "FINAL ANSWER:\n\n"
                + result.get("llm1_answer", "")
            )

        return "No final answer generated."

    except Exception as e:

        return f"ERROR:\n{str(e)}"


# ============================================================
# GRADIO
# ============================================================
demo = gr.Interface(
    fn=ask_ai,

    inputs=gr.Textbox(
        label="Question",
        placeholder="Enter your question..."
    ),

    outputs=gr.Textbox(
        label="Final Answer",
        lines=15
    ),

    title="3-LLM LangGraph System",

    description="""
LLM 1 → Generate Answer

LLM 2 → Check LLM 1

Correct → LLM 1 Final Answer

Incorrect → LLM 2 Corrected Answer

LLM 1 + LLM 2 Failure → LLM 3 Backup

LangSmith → Full LangGraph Tracing
"""
)


# ============================================================
# LAUNCH
# ============================================================
demo.launch(share=True)

API KEY STATUS
openrouter        : FOUND
GROQ_API_KEY      : FOUND
akshara           : FOUND
LANGSMITH_API_KEY : FOUND

LANGSMITH
Tracing: ENABLED
Project: 3-LLM-LangGraph-Generate-Check-Backup

LLMs
LLM 1: READY
LLM 2: READY
LLM 3: READY

BUILDING LANGGRAPH
✅ LANGGRAPH COMPILED SUCCESSFULLY
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://af0d24edc5220b4277.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================================
# 3-LLM LANGGRAPH + LANGSMITH TRACING + GRADIO
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL PACKAGES
# ------------------------------------------------------------
!pip install -q -U langgraph langchain-openrouter langchain-groq langsmith gradio


# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------
import os
import gradio as gr

from typing import TypedDict
from google.colab import userdata

from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END


# ============================================================
# 3. GET API KEYS FROM COLAB SECRETS
# ============================================================

OPENROUTER_API_KEY = userdata.get("openrouter")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
AKSHARA_API_KEY = userdata.get("akshara")
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")


# ============================================================
# 4. LANGSMITH CONFIGURATION
# ============================================================

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

# Enable LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# LangSmith project
os.environ["LANGCHAIN_PROJECT"] = (
    "3-LLM-LangGraph-Generate-Check-Backup"
)

# LangSmith endpoint
os.environ["LANGCHAIN_ENDPOINT"] = (
    "https://api.smith.langchain.com"
)


# ============================================================
# 5. CHECK API KEYS
# ============================================================

print()
print("=" * 55)
print("API KEY STATUS")
print("=" * 55)

print(
    f"openrouter        : "
    f"{'FOUND' if OPENROUTER_API_KEY else 'MISSING'}"
)

print(
    f"GROQ_API_KEY      : "
    f"{'FOUND' if GROQ_API_KEY else 'MISSING'}"
)

print(
    f"akshara           : "
    f"{'FOUND' if AKSHARA_API_KEY else 'MISSING'}"
)

print(
    f"LANGSMITH_API_KEY : "
    f"{'FOUND' if LANGSMITH_API_KEY else 'MISSING'}"
)


# ============================================================
# 6. CHECK LANGSMITH
# ============================================================

print()
print("=" * 55)
print("LANGSMITH")
print("=" * 55)

print("Tracing:", os.environ["LANGCHAIN_TRACING_V2"])
print("Project:", os.environ["LANGCHAIN_PROJECT"])
print("Endpoint:", os.environ["LANGCHAIN_ENDPOINT"])
print(
    "API Key:",
    "FOUND" if os.environ.get("LANGSMITH_API_KEY")
    else "MISSING"
)


# ============================================================
# 7. CREATE LLMs
# ============================================================

# ------------------------------------------------------------
# LLM 1
# Generate initial answer
# ------------------------------------------------------------

llm1 = ChatOpenRouter(
    model="openai/gpt-oss-20b",
    api_key=OPENROUTER_API_KEY,
    temperature=0
)


# ------------------------------------------------------------
# LLM 2
# Check LLM 1 answer
# ------------------------------------------------------------

llm2 = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)


# ------------------------------------------------------------
# LLM 3
# Emergency backup
# ------------------------------------------------------------

llm3 = ChatOpenRouter(
    model="qwen/qwen3-32b",
    api_key=AKSHARA_API_KEY,
    temperature=0
)


print()
print("LLMs")
print("LLM 1: READY")
print("LLM 2: READY")
print("LLM 3: READY")


# ============================================================
# 8. LANGGRAPH STATE
# ============================================================

class State(TypedDict, total=False):

    question: str

    llm1_answer: str
    llm2_answer: str
    llm3_answer: str

    check_result: str

    final_answer: str

    selected_llm: str

    error_llm1: str
    error_llm2: str
    error_llm3: str


# ============================================================
# 9. LLM 1 NODE
# ============================================================

def llm1_node(state: State):

    question = state["question"]

    try:

        # ----------------------------------------------------
        # FORCED FAILURE TEST
        # ----------------------------------------------------
        if "FORCE_LLM1_LLM2_FAILURE_TEST" in question:

            print("🔴 LLM 1: FORCED FAILURE")

            return {
                "llm1_answer": "",
                "error_llm1": "Forced test failure"
            }


        # ----------------------------------------------------
        # NORMAL LLM 1
        # ----------------------------------------------------

        print("🟢 LLM 1: GENERATING ANSWER")

        prompt = f"""
You are LLM 1.

Generate an accurate and clear answer
to the user's question.

Question:
{question}
"""

        response = llm1.invoke(prompt)

        answer = response.content

        if not answer:

            raise Exception(
                "LLM 1 returned an empty answer"
            )

        print("✅ LLM 1: ANSWER GENERATED")

        return {
            "llm1_answer": answer,
            "selected_llm": "LLM 1"
        }


    except Exception as e:

        print("❌ LLM 1 ERROR:", str(e))

        return {
            "llm1_answer": "",
            "error_llm1": str(e)
        }


# ============================================================
# 10. ROUTE AFTER LLM 1
# ============================================================

def route_after_llm1(state: State):

    # LLM 2 always receives control.
    # It either checks LLM 1 or handles LLM 1 failure.

    return "llm2"


# ============================================================
# 11. LLM 2 NODE
# ============================================================

def llm2_node(state: State):

    question = state["question"]

    llm1_answer = state.get(
        "llm1_answer",
        ""
    )


    try:

        # ----------------------------------------------------
        # FORCED FAILURE TEST
        # ----------------------------------------------------

        if "FORCE_LLM1_LLM2_FAILURE_TEST" in question:

            print("🔴 LLM 2: FORCED FAILURE")

            return {
                "llm2_answer": "",
                "check_result": "FAILED",
                "error_llm2": "Forced test failure"
            }


        # ----------------------------------------------------
        # IF LLM 1 FAILED
        # ----------------------------------------------------

        if not llm1_answer:

            print(
                "🟡 LLM 1 FAILED → "
                "LLM 2 ATTEMPTING ANSWER"
            )

            prompt = f"""
LLM 1 failed to answer the question.

You are LLM 2.

Answer the following question accurately.

Question:
{question}
"""

            response = llm2.invoke(prompt)

            answer = response.content

            if not answer:

                raise Exception(
                    "LLM 2 returned an empty answer"
                )

            print("✅ LLM 2: ANSWER GENERATED")

            return {
                "llm2_answer": answer,
                "check_result": "CORRECT",
                "selected_llm": "LLM 2"
            }


        # ----------------------------------------------------
        # NORMAL LLM 2 CHECK
        # ----------------------------------------------------

        print("🟢 LLM 2: CHECKING LLM 1 ANSWER")

        prompt = f"""
You are LLM 2, a strict answer checker.

Original Question:
{question}

LLM 1 Answer:
{llm1_answer}

Check whether LLM 1's answer is correct.

If the answer is correct:

Start your response with:
CORRECT

If the answer is incorrect:

Start your response with:
INCORRECT

Then provide the corrected answer.
"""

        response = llm2.invoke(prompt)

        result = response.content

        if not result:

            raise Exception(
                "LLM 2 returned an empty response"
            )


        # ----------------------------------------------------
        # CHECK RESULT
        # ----------------------------------------------------

        if result.strip().upper().startswith("CORRECT"):

            print("✅ LLM 2: LLM 1 ANSWER IS CORRECT")

            return {
                "llm2_answer": result,
                "check_result": "CORRECT"
            }


        else:

            print(
                "⚠️ LLM 2: LLM 1 ANSWER IS INCORRECT"
            )

            return {
                "llm2_answer": result,
                "check_result": "INCORRECT"
            }


    except Exception as e:

        print("❌ LLM 2 ERROR:", str(e))

        return {
            "llm2_answer": "",
            "check_result": "FAILED",
            "error_llm2": str(e)
        }


# ============================================================
# 12. ROUTE AFTER LLM 2
# ============================================================

def route_after_llm2(state: State):

    result = state.get(
        "check_result",
        ""
    )

    if result == "FAILED":

        print(
            "🔀 ROUTE: LLM 2 FAILED → LLM 3"
        )

        return "llm3"


    print(
        "🔀 ROUTE: LLM 2 SUCCESS → END"
    )

    return "end"


# ============================================================
# 13. LLM 3 BACKUP NODE
# ============================================================

def llm3_node(state: State):

    question = state["question"]

    try:

        print()
        print(
            "🚨 LLM 3: EMERGENCY BACKUP ACTIVATED"
        )

        prompt = f"""
You are LLM 3, the emergency backup AI.

LLM 1 and LLM 2 failed.

Provide the best possible accurate answer
to the following question.

Question:
{question}
"""

        response = llm3.invoke(prompt)

        answer = response.content

        if not answer:

            raise Exception(
                "LLM 3 returned an empty answer"
            )

        print("✅ LLM 3: BACKUP ANSWER GENERATED")

        return {
            "llm3_answer": answer,
            "final_answer": answer,
            "selected_llm": "LLM 3"
        }


    except Exception as e:

        print("❌ LLM 3 ERROR:", str(e))

        return {
            "final_answer":
                "All three LLMs failed.",
            "error_llm3": str(e),
            "selected_llm": "NONE"
        }


# ============================================================
# 14. BUILD LANGGRAPH
# ============================================================

print()
print("=" * 55)
print("BUILDING LANGGRAPH")
print("=" * 55)


graph = StateGraph(State)


# Add nodes
graph.add_node(
    "llm1",
    llm1_node
)

graph.add_node(
    "llm2",
    llm2_node
)

graph.add_node(
    "llm3",
    llm3_node
)


# START → LLM 1
graph.add_edge(
    START,
    "llm1"
)


# LLM 1 → LLM 2
graph.add_conditional_edges(
    "llm1",
    route_after_llm1,
    {
        "llm2": "llm2"
    }
)


# LLM 2 → END / LLM 3
graph.add_conditional_edges(
    "llm2",
    route_after_llm2,
    {
        "end": END,
        "llm3": "llm3"
    }
)


# LLM 3 → END
graph.add_edge(
    "llm3",
    END
)


# ============================================================
# 15. COMPILE GRAPH
# ============================================================

app = graph.compile()

print(
    "✅ LANGGRAPH COMPILED SUCCESSFULLY"
)


# ============================================================
# 16. ASK AI FUNCTION
# ============================================================

def ask_ai(question):

    if not question.strip():

        return (
            "Please enter a question."
        )


    print()
    print("=" * 55)
    print("NEW LANGGRAPH EXECUTION")
    print("=" * 55)

    print("Question:", question)


    try:

        # ----------------------------------------------------
        # INVOKE LANGGRAPH
        # ----------------------------------------------------

        result = app.invoke(
            {
                "question": question
            },

            config={
                "run_name":
                    "3-LLM-LangGraph-Generate-Check-Backup",

                "tags": [
                    "langgraph",
                    "3-llm",
                    "generate-check-backup"
                ],

                "metadata": {
                    "project":
                        "3-LLM-LangGraph-Generate-Check-Backup"
                }
            }
        )


        print()
        print("GRAPH EXECUTION COMPLETED")

        print(
            "Selected LLM:",
            result.get(
                "selected_llm",
                "UNKNOWN"
            )
        )


        # ----------------------------------------------------
        # LLM 3 BACKUP
        # ----------------------------------------------------

        if (
            result.get("selected_llm")
            == "LLM 3"
        ):

            return (
                "🚨 SELECTED LLM: "
                "LLM 3 - EMERGENCY BACKUP\n\n"

                "FINAL ANSWER:\n\n"

                + result.get(
                    "final_answer",
                    ""
                )
            )


        # ----------------------------------------------------
        # LLM 2 CORRECTED ANSWER
        # ----------------------------------------------------

        if (
            result.get("check_result")
            == "INCORRECT"
        ):

            return (
                "🟡 SELECTED LLM: LLM 2\n\n"

                "LLM 2 CORRECTION:\n\n"

                + result.get(
                    "llm2_answer",
                    ""
                )
            )


        # ----------------------------------------------------
        # LLM 1 ANSWER
        # ----------------------------------------------------

        if (
            result.get("check_result")
            == "CORRECT"
        ):

            # If LLM 2 generated the answer
            # because LLM 1 failed
            if (
                result.get("selected_llm")
                == "LLM 2"
            ):

                return (
                    "🟡 SELECTED LLM: LLM 2\n\n"

                    "FINAL ANSWER:\n\n"

                    + result.get(
                        "llm2_answer",
                        ""
                    )
                )


            return (
                "🟢 SELECTED LLM: LLM 1\n\n"

                "FINAL ANSWER:\n\n"

                + result.get(
                    "llm1_answer",
                    ""
                )
            )


        return (
            "No final answer generated."
        )


    except Exception as e:

        print(
            "❌ GRAPH ERROR:",
            str(e)
        )

        return (
            "ERROR:\n\n"
            + str(e)
        )


# ============================================================
# 17. GRADIO INTERFACE
# ============================================================

demo = gr.Interface(

    fn=ask_ai,

    inputs=gr.Textbox(
        label="Enter your question",

        placeholder=(
            "Example: What is "
            "artificial intelligence?"
        ),

        lines=3
    ),

    outputs=gr.Textbox(
        label="Final Answer",
        lines=15
    ),

    title=(
        "🤖 3-LLM LangGraph "
        "Generate → Check → Backup"
    ),

    description="""

### Architecture

**Question**
↓
**LLM 1 — Generate**
↓
**LLM 2 — Check**
↓
**Correct?**

YES → **LLM 1 Final Answer**

NO → **LLM 2 Corrected Answer**

If LLM 1 + LLM 2 fail
↓
**LLM 3 Emergency Backup**
↓
**Final Answer**

### LangSmith

Complete LangGraph execution is traced to:

**3-LLM-LangGraph-Generate-Check-Backup**

"""
)


# ============================================================
# 18. LAUNCH
# ============================================================

print()
print("=" * 55)
print("STARTING GRADIO")
print("=" * 55)

demo.launch(
    share=True,
    debug=True
)


API KEY STATUS
openrouter        : FOUND
GROQ_API_KEY      : FOUND
akshara           : FOUND
LANGSMITH_API_KEY : FOUND

LANGSMITH
Tracing: true
Project: 3-LLM-LangGraph-Generate-Check-Backup
Endpoint: https://api.smith.langchain.com
API Key: FOUND

LLMs
LLM 1: READY
LLM 2: READY
LLM 3: READY

BUILDING LANGGRAPH
✅ LANGGRAPH COMPILED SUCCESSFULLY

STARTING GRADIO
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bef1c72a49b8b10b00.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
